In [ ]:
"""Draw spot polygons (YOLO segmentation labels) on operculum images with a spot number on each spot."""
import os
import cv2
import numpy as np

from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "fish_ids.csv").is_file())
GROUP = "misc"
DATA = ROOT / "data" / GROUP
BASE = str(DATA)
IMG_DIR = os.path.join(BASE, "images", "normalized_operculums")
LBL_DIR = os.path.join(BASE, "labels", "spots")
OUT_DIR = os.path.join(BASE, "images", "matched")
os.makedirs(OUT_DIR, exist_ok=True)

done = 0
for fname in sorted(os.listdir(IMG_DIR)):
    stem, _ = os.path.splitext(fname)
    lbl_path = os.path.join(LBL_DIR, stem + ".txt")
    if not os.path.exists(lbl_path):
        print(f"No label for {fname}, skipped")
        continue

    img = cv2.imread(os.path.join(IMG_DIR, fname))
    h, w = img.shape[:2]
    scale = max(h, w) / 1000
    thick = max(1, round(2 * scale))
    font_scale = 0.7 * scale

    with open(lbl_path) as f:
        lines = [l.split() for l in f if l.strip()]

    for i, parts in enumerate(lines, start=1):
        coords = np.array(parts[1:], dtype=float).reshape(-1, 2)
        pts = np.round(coords * [w, h]).astype(np.int32)
        cv2.polylines(img, [pts], True, (0, 255, 0), thick)

        M = cv2.moments(pts)
        if M["m00"] != 0:
            cx, cy = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
        else:
            cx, cy = pts.mean(axis=0).astype(int)
        text = str(i)
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thick)
        org = (cx - tw // 2, cy + th // 2)
        cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), thick + 2, cv2.LINE_AA)
        cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 255, 255), thick, cv2.LINE_AA)

    cv2.imwrite(os.path.join(OUT_DIR, fname), img)
    done += 1

print(f"Saved {done} annotated images to {OUT_DIR}")
